In [28]:
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from torch.utils.data import Subset

In [29]:
cwd = Path.cwd()
home_dir = cwd.parent.parent
utils_path = home_dir / 'utils.py'
table_dir = home_dir / 'train val test tables'
model_dir = home_dir / 'models'

In [30]:
sys.path.append(str(home_dir))
import utils as ut

In [31]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_FEATURES = ['beta', 'Av', 'F115', 'F150', 'F277', 'F444']
# IN_FEATURES = ['F150', 'F277', 'F444']
OUT_FEATURES = ['flag']
SEED = 42

# HYPERPARAMETERS
EPOCHS = 10000
NUM_TRIALS = 100
PATIENCE = 30
N_SPLITS = 5
PRE_SCALING = None
OPTIMIZE = True

In [32]:
if OPTIMIZE:
    COMMENT = input('Enter OPTIMIZATION comment: ')

In [33]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ut.set_seeds(SEED)

In [34]:
# LOAD DATA

# loop through and read tables
table_dfs = [
    pd.read_csv(table, index_col=0)
    for table in table_dir.rglob('*')
    if table.name != 'test.csv'
]

# concat segmented dfs to train val sets
df = pd.concat(table_dfs)

len(pd.concat(table_dfs)), len(df)

(438, 438)

In [35]:
# FILTER & MASK

# filter for chosen features
df = df[IN_FEATURES + OUT_FEATURES]
df.dropna(inplace=True)

# mask unbounded results
fluxes = IN_FEATURES[2:]
df = df[~df[IN_FEATURES].eq(-99).any(axis=1)]

# replace negative fluxes with 0
df[fluxes] = df[fluxes].clip(lower=0)

len(df)

433

In [36]:
# PRE SCALE

fluxes = IN_FEATURES[2:]

if PRE_SCALING == 'log1p':
    df[fluxes] = np.log1p(df[fluxes])
    df[fluxes] = np.log1p(df[fluxes])

In [37]:
# CONVERT TO TENSORS AND DATASETS

inputs = torch.tensor(df[IN_FEATURES].values)
labels = torch.tensor(df['flag'].values)

dataset = torch.utils.data.TensorDataset(inputs, labels)

In [38]:
# DEFINE SCALING AND MODEL FUNCTIONS


def scale_data(X_train, X_misc):
    X_train = X_train.clone()
    X_misc = X_misc.clone()

    scaler = RobustScaler()
    # errors =  [item.replace("F", "E") for item in fluxes]

    scaled_train = scaler.fit_transform(X_train)
    scaled_misc = scaler.transform(X_misc)

    return scaled_train, scaled_misc


def define_model(trial):
    # We optimize the number of layers, hidden untis and dropout ratio in each layer.
    n_layers = trial.suggest_int('n_layers', 1, 3)
    layers = []

    in_features = len(IN_FEATURES)
    for i in range(n_layers):
        out_features = trial.suggest_int(f'n_units_l{i}', 4, 128)
        layers.append(nn.Linear(in_features, out_features, bias=False))
        layers.append(nn.BatchNorm1d(out_features))
        layers.append(nn.ReLU())
        p = trial.suggest_float(f'dropout_l{i}', 0.2, 0.5)
        layers.append(nn.Dropout(p))

        in_features = out_features
    layers.append(nn.Linear(in_features, 2 * len(OUT_FEATURES)))

    return nn.Sequential(*layers)

In [ ]:
# TRAINING LOOP FUNCTION


def objective(trial):
    global BEST_ACC

    kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_accuracies = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        model = define_model(trial).to(DEVICE)
        optimizer, loss_func, batch_size = ut.get_hyperparameters(
            trial, model, DEVICE
        )

        # get separate val and train features and labels and scale
        features, labels = dataset.tensors
        X_train, X_val = features[train_idx], features[val_idx]
        y_train, y_val = labels[train_idx], labels[val_idx]
        X_train, X_val = scale_data(X_train, X_val)

        # create subsets and dataloaders
        train_subset = Subset(X_train, y_train)
        val_subset = Subset(X_val, y_val)

        if SEED:
            train_loader = torch.utils.data.DataLoader(
                train_subset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=True,
            )
        else:
            train_loader = torch.utils.data.DataLoader(
                train_subset,
                batch_size=batch_size,
                shuffle=True,
                drop_last=True,
            )
        val_loader = torch.utils.data.DataLoader(
            val_subset, batch_size=batch_size, shuffle=False
        )

        # early stopper
        stopper = ut.EarlyStopper(patience=PATIENCE, min_delta=0.0005)
        for _ in range(EPOCHS):
            # Training Loop ---------------------------------------------------------------------------------------
            model.train()
            for data, labels in train_loader:
                data, labels = data.to(DEVICE), labels.to(DEVICE)

                optimizer.zero_grad()
                if data.ndim == 1:
                    data = data.unsqueeze(1)
                data = data.float()
                outputs = model(data)
                labels = labels.long()  # for calssification
                loss = loss_func(outputs, labels)

                loss.backward()
                optimizer.step()

            # Validation Loop --------------------------------------------------------------------------------------
            model.eval()
            running_loss = 0
            correct = 0
            with torch.no_grad():
                for data, labels in val_loader:
                    data, labels = data.to(DEVICE), labels.to(DEVICE)

                    if data.ndim == 1:
                        data = data.unsqueeze(1)
                    data = data.float()

                    outputs = model(data)
                    labels = labels.long()  # for calssification

                    loss = loss_func(outputs, labels)
                    running_loss += loss.item()

                    predictions = torch.argmax(outputs, dim=1)
                    correct += (predictions == labels).float().sum()

                # compute accuracy to report to optuna and check for pruning
                accuracy = correct / len(val_subset)
                if accuracy > BEST_ACC:
                    BEST_ACC = float(accuracy)

                # track loss once per epoch to check early stopper
                val_loss = running_loss / len(val_loader)
                stopper(val_loss)
                if stopper.early_stop:
                    break

        # report acc and check for pruning
        fold_accuracies.append(float(accuracy))
        trial.report(accuracy, fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy

In [40]:
# TRAINS MODEL

if OPTIMIZE:
    BEST_ACC = 0
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    sampler = optuna.samplers.TPESampler(seed=SEED)

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    )
    study.optimize(objective, n_trials=NUM_TRIALS, show_progress_bar=True)

    print('Best trial:')
    trial = study.best_trial
    print(f'  Accuracy: {trial.value}')
    print('  Params: ')
    for key, value in trial.params.items():
        print(f'    {key}: {value}')

  0%|          | 0/100 [00:00<?, ?it/s]

[W 2026-11-18 20:26:06,858] Trial 0 failed with parameters: {'n_layers': 2, 'n_units_l0': 122, 'dropout_l0': 0.4195981825434215, 'n_units_l1': 78, 'dropout_l1': 0.24680559213273096, 'lr': 4.207053950287933e-05, 'optimizer': 'SGD', 'sgd_momentum': 0.700991852018085, 'w1': 1.041168988591605, 'w2': 2.9398197043239884, 'batch_size': 16} because of the following error: ValueError('too many values to unpack (expected 2)').
Traceback (most recent call last):
  File "c:\Users\harty\miniconda3\envs\standard\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\harty\AppData\Local\Temp\ipykernel_12548\734208207.py", line 49, in objective
    for data, labels in train_loader:
        ^^^^^^^^^^^^
ValueError: too many values to unpack (expected 2)
[W 2026-11-18 20:26:06,859] Trial 0 failed with value None.


ValueError: too many values to unpack (expected 2)

In [ ]:
# LOG OPTIMIZATION

if OPTIMIZE:
    logged_params = trial.params.copy()
    logged_params['accuracy'] = trial.value
    logged_params['comment'] = COMMENT
    logged_params['in_features'] = IN_FEATURES
    logged_params['out_features'] = OUT_FEATURES
    logged_params['seed'] = SEED
    logged_params['epochs'] = EPOCHS
    logged_params['num_trials'] = NUM_TRIALS
    logged_params['patience'] = PATIENCE
    logged_params['best accuracy'] = BEST_ACC
    logged_params['N_folds'] = N_SPLITS

    logger = ut.RunLogger(filepath=home_dir / 'logs' / 'fc_log.json')
    logger.log_run(logged_params)